# Incoorporating new files

## Setup

Add your data path here (`base_path`)

In [2]:
# Add path
import sys
sys.path.append('../')

import importlib 
from util import config
from util import utilities as util
import util.data_loader as data_loader  # <-- add
from util.data_loader import DataLoader

import os
import shutil
import pandas as pd
from tqdm import tqdm

# -------------------------
# NEW DATA ROOT (only)
# -------------------------
NEW_ROOT = "/Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/new_videos"

# point metadata CSVs to NEW only
config.trial_info_path  = os.path.join(NEW_ROOT, "trial_info_20260511.csv")
config.animal_info_path = os.path.join(NEW_ROOT, "animal_info_20260511.csv")

# point data home to NEW only (important if DataLoader uses config.data_path)
config.data_path = NEW_ROOT

# reload util so it reads the updated config values
importlib.reload(util)
importlib.reload(data_loader)

# now this can ONLY read the NEW csvs
metadata_new = util.load_metadata(type="combined")

print("Using:")
print(" trial_info_path :", config.trial_info_path)
print(" animal_info_path:", config.animal_info_path)
print(" data_path       :", config.data_path)
print(" rows            :", len(metadata_new))

Using:
 trial_info_path : /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/new_videos/trial_info_20260511.csv
 animal_info_path: /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/new_videos/animal_info_20260511.csv
 data_path       : /Users/stella/Desktop/PhD/paulsen_lab/Analysis/kou_analysis/cheeseboard-data/new_videos
 rows            : 7256


## Utilities

In [3]:
def load_new_metadata():
    # util.load_metadata reads config.animal_info_path / config.trial_info_path
    return util.load_metadata(type="combined")

base_path = NEW_ROOT

def get_date_format(date):
    ''' Convert YYYYMMDD to "YYYY-MM-DD" '''
    date_str = str(date)
    date_formatted = f"{date_str[0:4]}-{date_str[4:6]}-{date_str[6:8]}"
    return date_formatted

def generate_source_file_path(dl, data_type = 'behVideo'):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    date_formatted = get_date_format(dl.date)
    date_folder_path = os.path.join(base_path, 'data', date_formatted)

    if data_type == 'triggerLoc':
        return os.path.join(date_folder_path, 'locations.csv')
    elif data_type == 'behVideo':
        return os.path.join(date_folder_path, 'movie', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}.avi")
    elif data_type == 'trackInfo':
        return os.path.join(date_folder_path, 'movie', 'tracking', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}_positions.csv")
    elif data_type == 'exampleImg':
        return os.path.join(date_folder_path, 'movie', 'tracking', f"{date_formatted}_{dl.id}_trial_{dl.trial_id}_trace.png")
    else:
        print('Unknown data type')

def generate_desti_file_path(dl, data_type = 'behVideo'):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    if data_type == 'triggerLoc':
        return dl.generate_filepath('triggerLoc', 'rawdata', 'behav', 'csv')
    if data_type == 'trackInfo':
        return dl.generate_filepath('position', 'derivatives', 'behav', 'csv')
    if data_type == 'exampleImg':
        return dl.generate_filepath('trace', 'derivatives', 'behav', 'png')
    if data_type == 'behVideo':
        return dl.generate_filepath('video', 'rawdata', 'behav', 'avi')
    else: 
        print('Unknown data type')

def check_source_file_exists(dl, data_type):
    '''
    Files for each trial:
        * trackInfo: `*.positions.csv`
        * exampleImg: `*.trace.png`
        * behVideo: `*.avi`
    Files for each day:
        * triggerLoc: `*.locations.csv`
    '''
    file = generate_source_file_path(dl, data_type)
    if not os.path.isfile(file):
        print(f'{dl.id}-{dl.date}-trial_{dl.trial_id}: {data_type} missing')
        return False
    return True


In [4]:
metadata = load_new_metadata()

for col in ['sub', 'ses', 'trial_id', 'date', 'trial_type_day']:
    if col in metadata.columns:
        metadata[col] = pd.to_numeric(metadata[col], errors='coerce').astype('Int64')

metadata.head()

,sub,id,date,ses,trial_id,trial_type,trial_type_day,test_type,usable,trial_note,genotype,test_order,remapping,short_duration,LFP,animal_note
0,1,A,20241205,1,1,H,1,NaN,1,NaN,ChR2,e,0,0,0,NaN
1,1,A,20241205,1,2,H,1,NaN,1,NaN,ChR2,e,0,0,0,NaN
2,1,A,20241205,1,3,H,1,NaN,1,NaN,ChR2,e,0,0,0,NaN
3,1,A,20241205,1,4,H,1,NaN,1,NaN,ChR2,e,0,0,0,NaN
4,1,A,20241205,1,5,H,1,NaN,1,NaN,ChR2,e,0,0,0,NaN


## Check existance of all files  

Files for each trial:  
    * trackInfo: `*.positions.csv`  
    * exampleImg: `*.trace.png`  
    * behVideo: `*.avi`  
Files for each day:  
    * triggerLoc: `*.locations.csv`  

In [5]:
data_types = {'trackInfo', 'exampleImg', 'behVideo', 'triggerLoc'}

flag = True
for rowIdx in range(len(metadata)):
    dl = DataLoader(metadata_ses=metadata.iloc[rowIdx])
    for data_type in data_types:
        if not check_source_file_exists(dl, data_type):
            flag = False

if flag:
    print('All source files exist!')

A-20241205-trial_1: trackInfo missing
A-20241205-trial_1: exampleImg missing
A-20241205-trial_1: triggerLoc missing
A-20241205-trial_1: behVideo missing
A-20241205-trial_2: trackInfo missing
A-20241205-trial_2: exampleImg missing
A-20241205-trial_2: triggerLoc missing
A-20241205-trial_2: behVideo missing
A-20241205-trial_3: trackInfo missing
A-20241205-trial_3: exampleImg missing
A-20241205-trial_3: triggerLoc missing
A-20241205-trial_3: behVideo missing
A-20241205-trial_4: trackInfo missing
A-20241205-trial_4: exampleImg missing
A-20241205-trial_4: triggerLoc missing
A-20241205-trial_4: behVideo missing
A-20241205-trial_5: trackInfo missing
A-20241205-trial_5: exampleImg missing
A-20241205-trial_5: triggerLoc missing
A-20241205-trial_5: behVideo missing
A-20241205-trial_6: trackInfo missing
A-20241205-trial_6: exampleImg missing
A-20241205-trial_6: triggerLoc missing
A-20241205-trial_6: behVideo missing
A-20241206-trial_1: trackInfo missing
A-20241206-trial_1: exampleImg missing
A-202

## Move data

In [6]:
for rowIdx in tqdm(range(len(metadata))):
    dl = DataLoader(metadata_ses=metadata.iloc[rowIdx])
    for data_type in data_types:
        source_file_path = generate_source_file_path(dl, data_type)
        desti_file_path = generate_desti_file_path(dl, data_type)

        os.makedirs(os.path.dirname(desti_file_path), exist_ok=True)

        if not os.path.isfile(source_file_path):
            print(f'{dl.id}-{dl.date}-trial_{dl.trial_id}: {data_type} missing')
            continue

        if data_type == 'triggerLoc':
            loc = pd.read_csv(source_file_path)
            loc = loc.rename(columns={
                'Animal': 'id',
                'Well_row': 'well_row',
                'Well_col': 'well_col',
                'Valence': 'valence',
            })
            loc = loc[loc['id'] == dl.id].reset_index(drop=True)
            loc.to_csv(desti_file_path, index=False)
        else:
            shutil.copy2(source_file_path, desti_file_path)
        

 19%|█▊        | 1355/7256 [00:00<00:00, 6844.99it/s]

A-20241205-trial_1: trackInfo missing
A-20241205-trial_1: exampleImg missing
A-20241205-trial_1: triggerLoc missing
A-20241205-trial_1: behVideo missing
A-20241205-trial_2: trackInfo missing
A-20241205-trial_2: exampleImg missing
A-20241205-trial_2: triggerLoc missing
A-20241205-trial_2: behVideo missing
A-20241205-trial_3: trackInfo missing
A-20241205-trial_3: exampleImg missing
A-20241205-trial_3: triggerLoc missing
A-20241205-trial_3: behVideo missing
A-20241205-trial_4: trackInfo missing
A-20241205-trial_4: exampleImg missing
A-20241205-trial_4: triggerLoc missing
A-20241205-trial_4: behVideo missing
A-20241205-trial_5: trackInfo missing
A-20241205-trial_5: exampleImg missing
A-20241205-trial_5: triggerLoc missing
A-20241205-trial_5: behVideo missing
A-20241205-trial_6: trackInfo missing
A-20241205-trial_6: exampleImg missing
A-20241205-trial_6: triggerLoc missing
A-20241205-trial_6: behVideo missing
A-20241206-trial_1: trackInfo missing
A-20241206-trial_1: exampleImg missing
A-202

 28%|██▊       | 2040/7256 [00:00<00:00, 6683.39it/s]

AJ-20250721-trial_1: trackInfo missing
AJ-20250721-trial_1: exampleImg missing
AJ-20250721-trial_1: triggerLoc missing
AJ-20250721-trial_1: behVideo missing
AJ-20250722-trial_1: trackInfo missing
AJ-20250722-trial_1: exampleImg missing
AJ-20250722-trial_1: triggerLoc missing
AJ-20250722-trial_1: behVideo missing
AJ-20250723-trial_1: trackInfo missing
AJ-20250723-trial_1: exampleImg missing
AJ-20250723-trial_1: triggerLoc missing
AJ-20250723-trial_1: behVideo missing
AJ-20250724-trial_1: trackInfo missing
AJ-20250724-trial_1: exampleImg missing
AJ-20250724-trial_1: triggerLoc missing
AJ-20250724-trial_1: behVideo missing
AJ-20250724-trial_1: trackInfo missing
AJ-20250724-trial_1: exampleImg missing
AJ-20250724-trial_1: triggerLoc missing
AJ-20250724-trial_1: behVideo missing
AJ-20250725-trial_1: trackInfo missing
AJ-20250725-trial_1: exampleImg missing
AJ-20250725-trial_1: triggerLoc missing
AJ-20250725-trial_1: behVideo missing
AJ-20250726-trial_1: trackInfo missing
AJ-20250726-trial_1

 38%|███▊      | 2726/7256 [00:00<00:00, 6743.26it/s]

...

 89%|████████▉ | 6467/7256 [00:01<00:00, 4735.13it/s]

N4-20260317-trial_4: exampleImg missing
N4-20260317-trial_4: triggerLoc missing
N4-20260317-trial_4: behVideo missing
N4-20260317-trial_5: trackInfo missing
N4-20260317-trial_5: exampleImg missing
N4-20260317-trial_5: triggerLoc missing
N4-20260317-trial_5: behVideo missing
N4-20260317-trial_6: trackInfo missing
N4-20260317-trial_6: exampleImg missing
N4-20260317-trial_6: triggerLoc missing
N4-20260317-trial_6: behVideo missing
N4-20260318-trial_1: trackInfo missing
N4-20260318-trial_1: exampleImg missing
N4-20260318-trial_1: triggerLoc missing
N4-20260318-trial_1: behVideo missing
N4-20260318-trial_2: trackInfo missing
N4-20260318-trial_2: exampleImg missing
N4-20260318-trial_2: triggerLoc missing
N4-20260318-trial_2: behVideo missing
N4-20260318-trial_3: trackInfo missing
N4-20260318-trial_3: exampleImg missing
N4-20260318-trial_3: triggerLoc missing
N4-20260318-trial_3: behVideo missing
N4-20260318-trial_4: trackInfo missing
N4-20260318-trial_4: exampleImg missing
N4-20260318-trial_

 97%|█████████▋| 7006/7256 [00:04<00:00, 620.84it/s] 

F6-20260430-trial_1: trackInfo missing
F6-20260430-trial_1: exampleImg missing


100%|██████████| 7256/7256 [00:06<00:00, 1125.21it/s]
